In [ ]:
# import sys
# sys.path.insert(0, "/home/ece-486/Documents/SP_PBL/pbl-lab/robot_ee_trajectories/ar_module")
from tools import tools_file
from tools import tools_plot

data_path = "Data_Processed/AR/ar_trajs_50Hz.pkl"
tf = tools_file.pkl_to_tf(data_path)

In [ ]:
from pathlib import Path
import pickle
import numpy as np

from tools import tools_file
from tools import tools_plot

TRAJ_PATH = Path("Data_Processed/AR/ar_trajs_50Hz.pkl")
EPISODE_ID = 0
SEED = 0
SCALE_RANGE = (0.85, 1.15)
NOISE_POS_STD = 0.03               # meters,  sampled at keyframes
NOISE_ORN_STD = np.deg2rad(10)     # radians, sampled at keyframes
NOISE_KEYFRAME_INTERVAL = 50      # frames

rng = np.random.default_rng(SEED)
with TRAJ_PATH.open("rb") as f:
    episodes = pickle.load(f)

episode = episodes[EPISODE_ID]
pos = np.asarray(episode["ee_pos"], dtype=np.float32)
rot = tools_file.axis_angle_to_rot(np.asarray(episode["ee_axis_angle"], dtype=np.float32))
pos_rel = pos - pos[:1]

scale = rng.uniform(*SCALE_RANGE)
scaled_pos = pos_rel * scale


def catmull_rom_keyframe_noise(num_steps, interval, std, rng):
    num_keyframes = ((num_steps - 1) // interval) + 2
    offsets = rng.standard_normal((num_keyframes, 3)).astype(np.float32) * std
    offsets[0] = 0.0

    step = np.arange(num_steps)
    left = step // interval
    right = np.minimum(left + 1, num_keyframes - 1)
    alpha = ((step - left * interval) / float(interval)).astype(np.float32)

    prev_key = np.maximum(left - 1, 0)
    next_key = np.minimum(right + 1, num_keyframes - 1)
    p0 = offsets[prev_key]
    p1 = offsets[left]
    p2 = offsets[right]
    p3 = offsets[next_key]
    alpha = alpha[:, None]
    alpha2 = alpha * alpha
    alpha3 = alpha2 * alpha
    return 0.5 * (
        (2.0 * p1)
        + (-p0 + p2) * alpha
        + (2.0 * p0 - 5.0 * p1 + 4.0 * p2 - p3) * alpha2
        + (-p0 + 3.0 * p1 - 3.0 * p2 + p3) * alpha3
    ).astype(np.float32)


def gaussian_smooth_noise(noise, interval):
    kernel_size = max(3, interval)
    if kernel_size % 2 == 0:
        kernel_size += 1
    radius = kernel_size // 2
    x = np.arange(kernel_size, dtype=np.float32) - radius
    sigma = max(float(kernel_size) / 6.0, 1.0)
    kernel = np.exp(-0.5 * (x / sigma) ** 2)
    kernel = kernel / kernel.sum()
    noise_padded = np.pad(noise, ((radius, radius), (0, 0)), mode="edge")
    return np.stack([
        np.convolve(noise_padded[:, axis], kernel, mode="valid")
        for axis in range(3)
    ], axis=1).astype(np.float32)


# 生成并平滑 position noise
noise = catmull_rom_keyframe_noise(len(pos_rel), NOISE_KEYFRAME_INTERVAL, NOISE_POS_STD, rng)
noise = gaussian_smooth_noise(noise, NOISE_KEYFRAME_INTERVAL)
noise = noise - noise[:1]

# 生成并平滑 orientation noise。这里的 3D noise 是局部 frame 下的 axis-angle 扰动。
orn_noise_axis_angle = catmull_rom_keyframe_noise(len(pos_rel), NOISE_KEYFRAME_INTERVAL, NOISE_ORN_STD, rng)
orn_noise_axis_angle = gaussian_smooth_noise(orn_noise_axis_angle, NOISE_KEYFRAME_INTERVAL)
orn_noise_axis_angle = orn_noise_axis_angle - orn_noise_axis_angle[:1]
orn_noise_rot = tools_file.axis_angle_to_rot(orn_noise_axis_angle)

# 叠加 noise
augmented_pos = scaled_pos + noise
augmented_rot = rot @ orn_noise_rot
original_tf   = tools_file.pos_rot_to_tf(pos_rel, rot)
noise_tf      = tools_file.pos_rot_to_tf(noise, orn_noise_rot)
augmented_tf  = tools_file.pos_rot_to_tf(augmented_pos, augmented_rot)


# 可视化 noise
ARROW_SIZE = 0.02
ARROW_INTV = 0.01

# original pos + original orientation
tools_plot.plot_3d(original_tf, ARROW_SIZE, ARROW_INTV, "normal")
tools_plot.plot_3d(augmented_tf, ARROW_SIZE, ARROW_INTV, "normal")


# augmented pos + augmented orientation
tools_plot.plot_3d(original_tf, ARROW_SIZE, ARROW_INTV, "top")
tools_plot.plot_3d(augmented_tf, ARROW_SIZE, ARROW_INTV, "top")

# position noise path + orientation noise axes
tools_plot.plot_3d(noise_tf, ARROW_SIZE/2, ARROW_INTV, "normal")
tools_plot.plot_3d(noise_tf, ARROW_SIZE/2, ARROW_INTV, "top")
